# Language Modelling 3
My notes for the following video by Andrej Karpathy:

- [_Let's build GPT: from scratch, in code, spelled out._](https://www.youtube.com/watch?v=kCc8FmEb1nY)

We finally implement NanoGPT, i.e. a simple model that uses transformers.

In [87]:
# SETUP AND DATA

import torch
import torch.nn as nn
from torch.nn import functional as F

# Hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 300
learning_rate = 1e-3
device = 'cpu' #'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32
# ------------------

# Import data
with open('tinyshakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# All unique characters in text
chars = sorted(list(set(text)))
vocab_size = len(chars)

# A simple "tokenizer": by-character encoding
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

# Data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # offsets
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval() # set mode
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones([block_size, block_size])))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        # Attention scores
        wei = q @ k.transpose(-2, -1) * C ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        # Weighted aggregation of values
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )
        
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = x.mean(1, keepdim=True)
        xvar = x.var(1, keepdim=True)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

# Model
class LanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            Block(n_embd, n_head = 4),
            Block(n_embd, n_head = 4),
            Block(n_embd, n_head = 4),
            nn.LayerNorm(n_embd)
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)

    # (B, T, C) batch time channel REMOVE THIS
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)
        x = self.blocks(x) # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)
        
        if targets is None:
            loss = None
        else: 
            B, T, C = logits.shape
            # PyTorch nees (B, C, T) for cross-entropy
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] # crop context
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # (B, C), only last time step
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = LanguageModel()
m = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    # Once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()        

step 0: train loss 4.3575, val loss 4.3599
step 300: train loss 2.5288, val loss 2.5494
step 600: train loss 2.3889, val loss 2.4066
step 900: train loss 2.2751, val loss 2.2958
step 1200: train loss 2.2259, val loss 2.2435
step 1500: train loss 2.1825, val loss 2.2108
step 1800: train loss 2.1519, val loss 2.1832
step 2100: train loss 2.1168, val loss 2.1523
step 2400: train loss 2.0974, val loss 2.1448
step 2700: train loss 2.0849, val loss 2.1152
step 3000: train loss 2.0738, val loss 2.1191
step 3300: train loss 2.0427, val loss 2.1068
step 3600: train loss 2.0318, val loss 2.1014
step 3900: train loss 2.0273, val loss 2.0820
step 4200: train loss 2.0129, val loss 2.0879
step 4500: train loss 1.9956, val loss 2.0802
step 4800: train loss 1.9904, val loss 2.0640


 ## Encoding

 Some words on encoding here.

## Training Examples

Make transform used to seeing contexts from length 1 to block_size 

In [27]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [56]:
# Start with newline character
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))


IChoo yofoofathesear avene BESANRito, tcond:
KINCULaued ad If lies t ngrond!
ICayer lle Andis?
DENUS


# Convolution trick

the token in the th position should only "talk" to previous tokens, i.e. tokens before it

calculate average of logit vectors of previous tokens

use matrix multiplication trick, lower left trick for averaging

In [70]:
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, 0) # (1, C)

# REPLACE WITH
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
#print(wei)
xbow2 = wei @ x # (T, T) @ (B, T, C) -> (B, T, T) @ (B, T, C) -> (B, T, C)

# 3rd version, use softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

# 4th version, self-attention
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
key = nn.Linear(C, head_size, bias=False) # matrix multiply
query = nn.Linear(C, head_size, bias=False) # matrix multiply
value = nn.Linear(C, head_size, bias=False) 
k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)
# note regarding stability of softmax because of variance, softmax converging to max if too big
wei = q @ k.transpose(-2, -1) * head_size ** -0.5# (B, T, 16) @ (B, 16, T) -> (B, T, T)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
v = value(x)
out = wei @ v


In [88]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))



3RIO:
He,O MOLHAMUCENES fleal in it by thing the glood,
ThatWITER:
SITHOMES:
And bath I with justleen not than this,
And aspeepling:
And the me, with fack fopenromm Rabsten, lade this tell poberter,
Themen, thee on'd
greclaif, Nor the ear.

GRIV:
That Sompard welman:
Though
Bunter lorth my to to such he pody.

YORKISS:
Nor Yains:
Go, gut on creet a repper.
:Aput Conguel:
The kitimbain that joing in pason!
Mance is have and, for lood his;
I warck hears now and mautake.

No-quollood riven parriti
